# Tutorial 03: Creating Custom PDPTW Problems

Learn how to create and customize your own Pickup and Delivery Problems with Time Windows (PDPTW) from scratch.

**What you'll learn:**
- Create PDPTW instances from scratch using manual data
- Load problem data from CSV files
- Customize node attributes (time windows, demands, service times)
- Validate problem feasibility

**Prerequisites:**
- Tutorial 01 (Quickstart)
- Basic Python knowledge

**Time:** ~40 minutes

## 1. Setup and Imports

In [ ]:
# Standard imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# VRP Toolkit imports
from vrp_toolkit.problems.pdptw import PDPTWInstance, Node
from vrp_toolkit.algorithms.alns.solver import ALNSSolver

# Verify imports
print("All imports successful!")

## 2. Quick Start: Simplest Possible Example

Let's start with the **simplest possible PDPTW instance**: one depot, one pickup, one delivery.

In [ ]:
# Create 3 nodes: depot (0), pickup (1), delivery (2)
nodes = [
    Node(node_id=0, x=0.0, y=0.0, node_type='depot'),
    Node(node_id=1, x=10.0, y=5.0, demand=1.0, time_window=(0, 100), service_time=5, node_type='pickup', pair_node_id=2),
    Node(node_id=2, x=20.0, y=10.0, demand=-1.0, time_window=(0, 100), service_time=5, node_type='delivery', pair_node_id=1)
]

# Create instance
simple_instance = PDPTWInstance(
    nodes=nodes,
    battery_capacity=100.0,
    max_route_time=200.0,
    vehicle_capacity=10.0
)

print(f"Created instance with {len(simple_instance.nodes)} nodes")
print(f"Number of pickup-delivery pairs: {simple_instance.n}")
print(f"Distance matrix shape: {simple_instance.distance_matrix.shape}")

**What just happened:**
- We created 3 Node objects: depot, pickup, delivery
- Pickup (node 1) and delivery (node 2) are paired via `pair_node_id`
- Pickup has positive demand (+1), delivery has negative demand (-1)
- PDPTWInstance automatically computed Euclidean distance matrix from (x, y) coordinates

Let's solve this simple instance:

In [ ]:
# Solve
solver = ALNSSolver()
solution = solver.solve(simple_instance)

print(f"Solution cost: {solution.objective_value():.2f}")
print(f"Routes: {solution.routes}")
print(f"Feasible: {solution.is_feasible()}")

## 3. Understanding Core Concepts

Now let's understand the key components of PDPTW instances.

### 3.1 Node Types

PDPTW problems have three node types:
- **Depot (0)**: Start and end location for vehicles
- **Pickup (odd IDs)**: Customer requests to pick up items
- **Delivery (even IDs)**: Locations to deliver items
- **Charging (optional)**: Battery recharging stations

In [ ]:
# Example: Create nodes with different types
depot = Node(node_id=0, x=0, y=0, node_type='depot')

pickup = Node(
    node_id=1, 
    x=10, y=5,
    demand=5.0,              # Positive for pickup
    time_window=(10, 50),    # Can visit between time 10-50
    service_time=3.0,        # Takes 3 time units to serve
    node_type='pickup',
    pair_node_id=2           # Paired with delivery node 2
)

delivery = Node(
    node_id=2,
    x=20, y=10,
    demand=-5.0,             # Negative for delivery (must match pickup)
    time_window=(20, 80),    # Different time window from pickup
    service_time=3.0,
    node_type='delivery',
    pair_node_id=1           # Paired with pickup node 1
)

charging = Node(
    node_id=5,
    x=15, y=7,
    node_type='charging',
    service_time=10.0        # Recharging takes time
)

print("Node types created successfully!")

### 3.2 Pickup-Delivery Pairing

**Critical rule:** Every pickup must have a corresponding delivery, and they must reference each other.

In [ ]:
# Create 2 pickup-delivery pairs
nodes_two_pairs = [
    Node(node_id=0, x=0, y=0, node_type='depot'),
    
    # Pair 1: nodes 1 (pickup) and 2 (delivery)
    Node(node_id=1, x=5, y=5, demand=3.0, time_window=(0, 100), service_time=2, node_type='pickup', pair_node_id=2),
    Node(node_id=2, x=15, y=10, demand=-3.0, time_window=(0, 100), service_time=2, node_type='delivery', pair_node_id=1),
    
    # Pair 2: nodes 3 (pickup) and 4 (delivery)
    Node(node_id=3, x=8, y=3, demand=4.0, time_window=(0, 100), service_time=2, node_type='pickup', pair_node_id=4),
    Node(node_id=4, x=12, y=12, demand=-4.0, time_window=(0, 100), service_time=2, node_type='delivery', pair_node_id=3),
]

instance_two_pairs = PDPTWInstance(
    nodes=nodes_two_pairs,
    battery_capacity=100.0,
    max_route_time=200.0,
    vehicle_capacity=10.0
)

print(f"Created instance with {instance_two_pairs.n} pickup-delivery pairs")
print(f"Pickup-delivery pairs: {instance_two_pairs.pickup_delivery_pairs}")

### 3.3 Distance Matrix

The distance matrix defines travel distances between all nodes. By default, it's computed from Euclidean coordinates.

In [ ]:
# View the distance matrix
print("Distance matrix (Euclidean):")
print(instance_two_pairs.distance_matrix)
print(f"\nShape: {instance_two_pairs.distance_matrix.shape}")
print(f"Distance from depot (0) to pickup 1 (1): {instance_two_pairs.distance_matrix[0, 1]:.2f}")

You can also provide a **custom distance matrix** (e.g., from real road networks):

In [ ]:
# Create custom distance matrix (e.g., from road network)
custom_distances = np.array([
    [0,    10,   25,   12,   30],  # From depot
    [10,   0,    18,   8,    22],  # From pickup 1
    [25,   18,   0,    20,   15],  # From delivery 1
    [12,   8,    20,   0,    28],  # From pickup 2
    [30,   22,   15,   28,   0]    # From delivery 2
])

# Create instance with custom distances
instance_custom = PDPTWInstance(
    nodes=nodes_two_pairs,
    battery_capacity=100.0,
    max_route_time=200.0,
    vehicle_capacity=10.0
)

# Override distance matrix
instance_custom.distance_matrix = custom_distances

print("Custom distance matrix applied!")
print(f"Distance from depot to pickup 1: {instance_custom.distance_matrix[0, 1]:.2f}")

## 4. Advanced Features

### 4.1 Loading from CSV

**When to use:** Working with data from external sources (databases, Excel, etc.)

**How it works:** Create a pandas DataFrame with node information, then convert to PDPTWInstance.

In [ ]:
# Create sample CSV data
csv_data = """
node_id,type,x,y,demand,tw_start,tw_end,service_time,pair_id
0,depot,0.0,0.0,0.0,0,200,0,0
1,cp,10.0,5.0,2.5,10,60,3,2
2,cd,20.0,8.0,-2.5,30,80,3,1
3,cp,8.0,12.0,3.0,15,70,4,4
4,cd,18.0,15.0,-3.0,40,90,4,3
"""

# Save to file
import io
df = pd.read_csv(io.StringIO(csv_data))
df.to_csv('sample_problem.csv', index=False)

print("CSV data:")
print(df)

In [ ]:
# Load from CSV
order_table = pd.read_csv('sample_problem.csv')

# Create instance from DataFrame
instance_from_csv = PDPTWInstance(
    order_table=order_table,
    battery_capacity=100.0,
    max_route_time=200.0,
    vehicle_capacity=10.0
)

print(f"Loaded instance with {instance_from_csv.n} pickup-delivery pairs")
print(f"Nodes: {len(instance_from_csv.nodes)}")

### 4.2 Time Windows and Feasibility

**When to use:** Modeling real-world scheduling constraints (delivery windows, business hours, etc.)

**How it works:** Each node has `time_window=(earliest, latest)` specifying when service can begin.

In [ ]:
# Example: Tight time windows (harder problem)
nodes_tight_tw = [
    Node(node_id=0, x=0, y=0, node_type='depot'),
    
    # Morning delivery: must pick up 8-10am, deliver 9-11am
    Node(node_id=1, x=10, y=5, demand=2.0, time_window=(8, 10), service_time=0.5, node_type='pickup', pair_node_id=2),
    Node(node_id=2, x=20, y=10, demand=-2.0, time_window=(9, 11), service_time=0.5, node_type='delivery', pair_node_id=1),
    
    # Afternoon delivery: must pick up 2-4pm, deliver 3-5pm
    Node(node_id=3, x=5, y=15, demand=3.0, time_window=(14, 16), service_time=0.5, node_type='pickup', pair_node_id=4),
    Node(node_id=4, x=15, y=20, demand=-3.0, time_window=(15, 17), service_time=0.5, node_type='delivery', pair_node_id=3),
]

instance_tight = PDPTWInstance(
    nodes=nodes_tight_tw,
    battery_capacity=100.0,
    max_route_time=480.0,  # 8-hour shift
    vehicle_capacity=10.0
)

print("Created instance with tight time windows")
print("Time windows:")
for node in instance_tight.nodes[1:]:
    print(f"  Node {node.node_id} ({node.node_type}): {node.time_window}")

### 4.3 Charging Stations

**When to use:** Modeling electric vehicles with battery constraints

**How it works:** Add nodes with `node_type='charging'` where vehicles can recharge.

In [ ]:
# Example: Problem with charging station
nodes_with_charging = [
    Node(node_id=0, x=0, y=0, node_type='depot'),
    
    # Far pickup-delivery pair
    Node(node_id=1, x=30, y=20, demand=2.0, time_window=(0, 100), service_time=2, node_type='pickup', pair_node_id=2),
    Node(node_id=2, x=50, y=40, demand=-2.0, time_window=(0, 100), service_time=2, node_type='delivery', pair_node_id=1),
    
    # Charging station in between
    Node(node_id=5, x=25, y=25, node_type='charging', service_time=5.0)
]

instance_charging = PDPTWInstance(
    nodes=nodes_with_charging,
    battery_capacity=50.0,  # Limited battery
    max_route_time=200.0,
    vehicle_capacity=10.0
)

print(f"Created instance with {len(instance_charging.charging_stations)} charging station")
print(f"Charging station at: ({nodes_with_charging[3].x}, {nodes_with_charging[3].y})")

## 5. Real-World Example: Campus Food Delivery

Let's create a realistic campus food delivery scenario:
- Central kitchen (depot)
- 3 pickup locations (restaurants)
- 3 delivery locations (dorms)
- Lunch time windows (11am-2pm)

In [ ]:
# Campus food delivery problem
campus_nodes = [
    # Central kitchen
    Node(node_id=0, x=0, y=0, node_type='depot'),
    
    # Order 1: Pizza from restaurant A to dorm 1
    Node(node_id=1, x=5, y=3, demand=2.0, time_window=(11, 13), service_time=0.25, node_type='pickup', pair_node_id=2),
    Node(node_id=2, x=12, y=8, demand=-2.0, time_window=(11.5, 13.5), service_time=0.25, node_type='delivery', pair_node_id=1),
    
    # Order 2: Sushi from restaurant B to dorm 2
    Node(node_id=3, x=8, y=2, demand=1.5, time_window=(11.5, 13.5), service_time=0.25, node_type='pickup', pair_node_id=4),
    Node(node_id=4, x=10, y=12, demand=-1.5, time_window=(12, 14), service_time=0.25, node_type='delivery', pair_node_id=3),
    
    # Order 3: Burgers from restaurant C to dorm 3
    Node(node_id=5, x=3, y=7, demand=3.0, time_window=(11, 12.5), service_time=0.25, node_type='pickup', pair_node_id=6),
    Node(node_id=6, x=15, y=5, demand=-3.0, time_window=(11.5, 13), service_time=0.25, node_type='delivery', pair_node_id=5),
]

campus_delivery = PDPTWInstance(
    nodes=campus_nodes,
    battery_capacity=100.0,
    max_route_time=180.0,  # 3-hour shift
    vehicle_capacity=10.0   # Can carry 10 units of food
)

print("Campus food delivery instance created!")
print(f"Orders: {campus_delivery.n}")
print(f"Total nodes: {len(campus_delivery.nodes)}")

In [ ]:
# Solve the campus delivery problem
solver = ALNSSolver()
campus_solution = solver.solve(campus_delivery)

print(f"\nSolution found!")
print(f"Total distance: {campus_solution.objective_value():.2f} units")
print(f"Feasible: {campus_solution.is_feasible()}")
print(f"\nRoutes:")
for i, route in enumerate(campus_solution.routes, 1):
    print(f"  Vehicle {i}: {route}")

In [ ]:
# Visualize the problem and solution
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# Plot 1: Problem layout
ax1.scatter([campus_nodes[0].x], [campus_nodes[0].y], c='red', s=200, marker='s', label='Depot', zorder=3)
pickup_nodes = [n for n in campus_nodes if n.node_type == 'pickup']
delivery_nodes = [n for n in campus_nodes if n.node_type == 'delivery']
ax1.scatter([n.x for n in pickup_nodes], [n.y for n in pickup_nodes], c='blue', s=100, marker='^', label='Pickup', zorder=3)
ax1.scatter([n.x for n in delivery_nodes], [n.y for n in delivery_nodes], c='green', s=100, marker='v', label='Delivery', zorder=3)

# Draw pickup-delivery pairs
for pickup in pickup_nodes:
    delivery = campus_nodes[pickup.pair_node_id]
    ax1.plot([pickup.x, delivery.x], [pickup.y, delivery.y], 'k--', alpha=0.3, zorder=1)

ax1.set_xlabel('X Coordinate')
ax1.set_ylabel('Y Coordinate')
ax1.set_title('Campus Food Delivery Problem')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Plot 2: Solution routes
ax2.scatter([campus_nodes[0].x], [campus_nodes[0].y], c='red', s=200, marker='s', label='Depot', zorder=3)
ax2.scatter([n.x for n in pickup_nodes], [n.y for n in pickup_nodes], c='blue', s=100, marker='^', label='Pickup', zorder=3)
ax2.scatter([n.x for n in delivery_nodes], [n.y for n in delivery_nodes], c='green', s=100, marker='v', label='Delivery', zorder=3)

# Draw solution routes
colors = ['purple', 'orange', 'brown']
for route_idx, route in enumerate(campus_solution.routes):
    route_coords = [(campus_nodes[node_id].x, campus_nodes[node_id].y) for node_id in route]
    xs, ys = zip(*route_coords)
    ax2.plot(xs, ys, color=colors[route_idx % len(colors)], linewidth=2, marker='o', label=f'Route {route_idx+1}', zorder=2)

ax2.set_xlabel('X Coordinate')
ax2.set_ylabel('Y Coordinate')
ax2.set_title(f'Solution (Cost: {campus_solution.objective_value():.2f})')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

**Key observations:**
- Pickup nodes (restaurants) are grouped by location
- Delivery nodes (dorms) are spread across campus
- Solution respects pickup-before-delivery constraints
- Time windows ensure food arrives during lunch hours

## 6. Comparison and Best Practices

**When to create problems manually:**
- Small test cases for algorithm development
- Teaching and demonstrations
- Debugging specific scenarios

**When to load from CSV:**
- Real-world data from databases or spreadsheets
- Large instances (>20 orders)
- Reproducible benchmarking

**When to use generators:**
- Synthetic testing data
- Algorithm benchmarking
- Parameter sensitivity analysis
- See Tutorial 07 for details

**When to use OSMnx integration:**
- Real street network distances
- Geographic accuracy matters
- Urban delivery scenarios
- See Tutorial 02 for details

**Common pitfalls:**
- **Mismatched pair IDs:** Ensure pickup.pair_node_id == delivery.node_id and vice versa
- **Demand mismatch:** Pickup demand must be positive, delivery must be negative of same magnitude
- **Infeasible time windows:** Delivery time window must allow arrival after pickup + travel time
- **Distance matrix size:** Must be n×n where n = number of nodes

## 7. Practice Exercises

Try these on your own:

1. **Basic:** Create a 2-order PDPTW instance with very tight time windows (1-hour windows). Can ALNS still find a solution?

2. **Intermediate:** Load the campus delivery problem from a CSV file you create. Modify the time windows to simulate dinner delivery (5pm-8pm).

3. **Advanced:** Create a problem with 5 orders where some orders have overlapping time windows and require 2 vehicles. Add a charging station and set battery capacity low enough that vehicles must recharge.

**Hints:**
- For Exercise 1: Use time_window=(0, 1) or similar
- For Exercise 2: Convert the campus_nodes to a DataFrame first
- For Exercise 3: Calculate total route distance and set battery_capacity < max_route_distance

In [ ]:
# Your solutions here


## 8. Summary

**What you learned:**
- ✅ Create PDPTW instances from Node objects
- ✅ Define pickup-delivery pairs with proper pairing
- ✅ Set time windows and service times
- ✅ Load problems from CSV files
- ✅ Use custom distance matrices
- ✅ Add charging stations for battery constraints

**Key takeaways:**
1. Every pickup must have exactly one paired delivery with opposite demand
2. Time windows define earliest and latest service start times
3. Distance matrix can be Euclidean (default) or custom (e.g., road network)
4. Charging stations enable modeling of electric vehicle constraints

**Next steps:**
- Try **Tutorial 04: Problem Variants** to learn about VRP, PDP, and CVRP variations
- Try **Tutorial 02: Real-World Maps** to use OSMnx for realistic distance matrices
- Try **Tutorial 06: Custom Algorithms** to implement your own solving methods
- Explore **Tutorial 07: Data Generation** for synthetic problem generation